# 2022 -Geodaten für Updates zu «Gang dur Züri» aufbereiten
Datum: 28.02.2022

Mit Colab kann **dieses Jupyter-Notebook interaktiv im Browser** gestartet werden:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DonGoginho/myPy/blob/main/update_ogd/gang_dur_zueri/gang_dur_zueri_geodaten_aufbereiten_2022.ipynb)


Die QGIS-Projekte für die Quartierspiegel sind unter 
- `O:\GIS\Sandbox\*JAHR*\GdZ` abgelegt.

Dieses Notebook hilft die aktiven Layer – die für die Karte vom [**Geoserver**](https://www.stadt-zuerich.ch/geodaten/) als View bezogen sind – als shp zu exportieren.

<!---  
Dieses Notebook verwendet als Vorlage die Webseite «[Kitas in Berlin - Jupyter Notebook](https://juanitorduz.github.io/kitas_berlin/)» 
---!>

## Importiere notwendige Packages:

In [1]:
#pip install geopandas altair fiona requests folium mplleaflet contextily seaborn datetime

In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import altair as alt
import fiona
import requests
import mplleaflet
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from datetime import datetime


## Datenaufbereitung

### Globale Variabeln

Als Zeitvariable berechnen wir erst mal das heutige Datum. Wir können diese Werte als Dateinamen verwenden.

**Steuerungsvariable activate_date**: 
- False: in den Filenamen kommen beim Export KEINE Datenangaben mit
- True: die Exportfiles haben *FILENAMEN_DATUM* im Namen

In [3]:
activate_date = False;


#https://www.programiz.com/python-programming/datetime/strftime
now = datetime.now() # current date and time
year_today = now.strftime("%Y")

if activate_date == True:
    date_today = "_"+now.strftime("%Y-%m-%d")
    #store_o_gis = False
    
else:
    date_today =""
    #store_o_gis = True


In [4]:
print("Datum: "+date_today +" Jahr: "+year_today)

Datum:  Jahr: 2022


### Pfade


In [5]:
gis_ablagepfad = "//szh/ssz/data/GIS/Daten/Vektor/"+year_today+"/GdZ/"
print(gis_ablagepfad )



//szh/ssz/data/GIS/Daten/Vektor/2022/GdZ/


### Basis Geodaten

Für grundsätzliche räumliche Prüfungen importieren wir hier auch die 
- **Statistischen Quartiere**
- **Stadtgrenze** (aus der SSZ-Geodatenablage)



In [6]:
# Read abt. Geopandas https://geopandas.org/docs/user_guide/io.html
url0 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Statistische_Quartiere?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=adm_statistische_quartiere_map"
gdf_adm_statistische_quartiere_map = gpd.read_file(url0)
print('done')

done


In [7]:
# Read abt. Geopandas https://geopandas.org/docs/user_guide/io.html
pfad_grenze = r"\\szh\ssz\data\GIS\Daten\Vektor\_aktuell\adm_stadtgrenze_a.shp"
gdf_adm_stadtgrenze_lv95 = gpd.read_file(pfad_grenze)

gdf_adm_stadtgrenze_wgs84_all =gdf_adm_stadtgrenze_lv95.to_crs(epsg=4326)

print('done')


done


In [8]:
# Rename columns bei Bedarf
rename_cols = {
    'name': 'bfs_name',
}

columns_to_drop = ['objid']

gdf_adm_stadtgrenze_wgs84 = gdf_adm_stadtgrenze_wgs84_all \
    .copy() \
    .drop(columns_to_drop, axis=1) \
    .rename(columns=rename_cols) \
    

In [9]:
#gdf_adm_stadtgrenze_wgs84

## Geodaten für Gang dur Züri aufbereiten

Beziehe die notwendigen Geodaten vom Geoportal und exportiere sie als Shapefiles.


### Baumkataster (1)

In [97]:
#Step1
url1 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Baumkataster?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=baumkataster_baumstandorte"

#Step2
typename="baumkataster_baumstandorte"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

#Step 3
imp_file = gpd.read_file(url1)

imp_file.info()
#imp_file['bezeichnung'].unique()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 75720 entries, 0 to 75719
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   id                 75720 non-null  object  
 1   baumartlat         74463 non-null  object  
 2   baumgattunglat     75720 non-null  object  
 3   baumnamedeu        75720 non-null  object  
 4   baumnamelat        75720 non-null  object  
 5   baumnummer         75720 non-null  object  
 6   baumtyp            75037 non-null  float64 
 7   baumtyptext        75720 non-null  object  
 8   genauigkeit        75720 non-null  object  
 9   kategorie          75720 non-null  object  
 10  kronendurchmesser  75720 non-null  int64   
 11  objid              75720 non-null  object  
 12  pflanzjahr         58488 non-null  float64 
 13  poi_id             75720 non-null  object  
 14  quartier           75720 non-null  object  
 15  status             75720 non-null  object  
 

In [99]:
#myAgg = data2betested.loc["2008-11-30":"2021-10-31"]
myAgg = imp_file\
    .groupby(['kategorie']) \
    .agg(count=('id', 'count')) \
    .sort_values('count', ascending=False) 

myAgg.reset_index().head(30)

,kategorie,count
0,Parkbaum,53119
1,Strassenbaum,22601


#### Step2: Datensätze exportieren


In [100]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\*JAHR*\quartierspiegel` als Shapefile


In [101]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/GdZ/export_baumkataster_baumstandorte.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [102]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_baumkataster_baumstandorte.gpkg


### Beachvolley (3)

#### Step 1: Daten vom Geoportal beziehen und prüfen.

In [26]:
url3 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_beachvolleyball_view"

typename="poi_beachvolleyball_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

#platziere urlX
imp_file = gpd.read_file(url3)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_beachvolleyball_view_wgs84.head(2)

array(['Schulhaus Stettbach', 'Oberer Letten', 'Schulhaus Im Lee',
       'Schulhaus Buhnrain', 'Freibad Zwischen den Hölzern',
       'Sportanlage Juchof 1', 'Josefwiese', 'Schulhaus Letzi',
       'Sportzentrum Sihlhölzli', 'Schulhaus Lachenzelg',
       'Freibad Seebach', 'Sportzentrum Buchlern', 'Freibad Allenmoos',
       'Sportzentrum Fronwald'], dtype=object)

#### Step2: Datensätze exportieren


In [27]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_beachvolleyball_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [28]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/GdZ/export_poi_beachvolleyball_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [29]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_beachvolleyball_view.gpkg


### Freibad

In [30]:
url7 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_freibad_view"

typename="poi_freibad_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

#place urlX
imp_file = gpd.read_file(url7)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Freibad Letzigraben', 'Freibad Dolder',
       'Freibad Zwischen den Hölzern', 'Freibad Allenmoos',
       'Freibad Auhof', 'Freibad Heuried', 'Freibad Seebach'],
      dtype=object)

#### Step2: Datensätze exportieren


In [31]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [32]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/GdZ/export_poi_freibad_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [33]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_freibad_view.gpkg


### Fussballplätze (8)

In [34]:
url8 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_fussballplatz_view"

typename="poi_fussballplatz_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url8)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Rasensportanlage Katzenbach', 'Sportzentrum Hardhof',
       'Rasensportanlage Juchhof 1', 'Sportzentrum Utogrund',
       'Sportzentrum Eichrain', 'Sportzentrum Fronwald',
       'Sportzentrum Heerenschürli', 'Sportzentrum Witikon',
       'Rasensportanlage Neudorf', 'Sportzentrum Heuried',
       'Rasensportanlage Juchhof 2', 'Rasensportanlage Allmend Brunau',
       'Sportzentrum Buchlern', 'Rasensportanlage Forrenweid',
       'Rasenspielfeld Liguster'], dtype=object)

#### Step2: Datensätze exportieren


In [35]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [36]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/GdZ/export_poi_fussballplatz_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [37]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_fussballplatz_view.gpkg


### Kindertagesstätten (9)

In [43]:
url9 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Kindertagesstaette?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_kindertagesstaette_view"

typename="poi_kindertagesstaette_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url9)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Kindertagesstätte USZ Culmannstrasse',
       'globegarden Bellerivestrasse', 'globegarden Limmattalstrasse',
       'Tandem IMS', 'Kita Chinderstern Zürich-Altstetten', 'KITA BÄRLIS',
       'Kita Mattenhof', 'Kinderkrippe Minimuus',
       'globegarden Münchhalden', 'Alexi`s Kinderkrippe (Bürglistrasse)',
       'Chinderhus Teddybär', 'Chinderhuus Babar',
       'Kita Schmetterling Bachwiesen',
       'Sunshine Kinderkrippe Zürichberg', 'Kinderkrippe Wallaby',
       'MIC - Montessori Infant Community Belvoir', 'Kita Sternenstaub',
       'Kita Gwunderfitz für kids', 'KiTa Little Angels',
       'Kita Leutschenbach', 'Güxi Norastrasse', 'Kinderkrippe Operinos',
       'Kinderkrippe Luna', 'Kinderkrippe Murmel', 'Krippe Heizenholz',
       'Chindsgi Hönggerberg', 'Kinderkrippe Hoppel', 'KIMI Central',
       'Kita In Böden', 'Zentrum Inselhof - Kindertagesstätte',
       'Kita Käferberg', 'Kita Tièchestrasse', 'KiTa KiBiZ Altstetten',
       'Villa Kunterbunt am Letzigraben Ki

#### Step2: Datensätze exportieren


In [44]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [45]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/GdZ/export_poi_kindertagesstaette_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [46]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_kindertagesstaette_view.gpkg


### Volksschulen (10)

In [48]:
url10 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Schulanlagen?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_volksschule_view"

typename="poi_volksschule_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url10)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Döltschi', 'Stettbach', 'Mattenhof', 'Neubühl', 'Letzi',
       'Loogarten', 'Chriesiweg', 'Feldblumen', 'Feld', 'Vogtsrain',
       'Freiestrasse', 'Hofacker Primar', 'Hofacker Sek',
       'Friedrichstrasse', 'Tagesschule Himmeri-Heumatt', 'Friesenberg',
       'Hürstholz', 'Seefeld', 'Sihlfeld', 'Kolbenacker', 'Scherr',
       'Probstei', 'Manegg', 'Saatlen', 'Triemli', 'Turner', 'Leimbach',
       'Holderbach', 'Rütihof', 'Luchswiesen', 'Grünau', 'Milchbuck',
       'Im Gut', 'Rebhügel', 'Langmatt', 'Schütze', 'Herzogenmühle',
       'Auhof', 'Heubeeribüel', 'Tagesschule Himmeri-Staudenbühl',
       'Hirschengraben', 'Fluntern', 'Bungertwies',
       'Kunst- & Sportschule Zürich', 'Hutten', 'Ilgen A', 'Ilgen B',
       'Leutschenbach', 'Im Birch', 'Limmat A', 'Hirzenbach', 'Kügeliloo',
       'Holderbach 2 - Pavillon Ruggächer', 'Münchhalde Primar',
       'Mühlebach', 'Münchhalde Sek', 'Kappeli Sekundar', 'Neumünster',
       'Albisriederplatz', 'Nordstrasse', 'Auzelg II',

#### Step2: Datensätze exportieren


In [49]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\*JAHR*\quartierspiegel` als Shapefile


In [50]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/GdZ/export_poi_volksschule_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [51]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_volksschule_view.gpkg


### Brunnen (11)

In [70]:
#Step1
url11 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Brunnen?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=wvz_brunnen"

#Step2
typename="wvz_brunnen"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

#Step 3
imp_file = gpd.read_file(url11)

imp_file.info()
#imp_file['bezeichnung'].unique()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1280 entries, 0 to 1279
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   id                    1280 non-null   object  
 1   art_txt               1280 non-null   object  
 2   baujahr               865 non-null    float64 
 3   betreiber_txt         1277 non-null   object  
 4   bezeichnung           771 non-null    object  
 5   brunnenart_txt        1280 non-null   object  
 6   brunnennummer         1206 non-null   object  
 7   druckzone             0 non-null      object  
 8   druckzone_txt         1278 non-null   object  
 9   eigentuemer_txt       1278 non-null   object  
 10  historisches_baujahr  997 non-null    float64 
 11  nummer                1280 non-null   object  
 12  objectid              1280 non-null   int64   
 13  wasserart_txt         1280 non-null   object  
 14  geometry              1280 non-null   geometry
d

In [77]:
#myAgg = data2betested.loc["2008-11-30":"2021-10-31"]
myAgg = imp_file\
    .groupby(['wasserart_txt']) \
    .agg(count=('id', 'count')) \
    .sort_values('count', ascending=False) 

myAgg.reset_index().head(30)

,wasserart_txt,count
0,Verteilnetz,814
1,Quellwasser,430
2,eigene Versorgung,31
3,Grundwasser,5


#### Step2: Datensätze exportieren


In [78]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\*JAHR*\quartierspiegel` als Shapefile


In [79]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/GdZ/export_wvz_brunnen.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [80]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_wvz_brunnen.gpkg


### Zweiradabstellplatz (12)

In [81]:
#Step1
url12 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Zweiradparkierung?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=zweiradabstellplaetze_p"

#Step2
typename="zweiradabstellplaetze_p"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

#Step 3
imp_file = gpd.read_file(url12)

imp_file.info()
#imp_file['bezeichnung'].unique()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1949 entries, 0 to 1948
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   id                   1949 non-null   object  
 1   anzahl_pp            1949 non-null   float64 
 2   bemerkung            242 non-null    object  
 3   dach                 0 non-null      object  
 4   gebuehr              1949 non-null   object  
 5   kategorie            1949 non-null   object  
 6   markierung           0 non-null      object  
 7   name                 1949 non-null   object  
 8   objectid             1949 non-null   float64 
 9   oeffentlicher_grund  1949 non-null   object  
 10  poi_id               1949 non-null   object  
 11  sicherheit           0 non-null      object  
 12  signalisation        0 non-null      object  
 13  geometry             1949 non-null   geometry
dtypes: float64(2), geometry(1), object(11)
memory usage: 213.3+ KB


In [93]:
#myAgg = data2betested.loc["2008-11-30":"2021-10-31"]
myAgg = imp_file\
    .groupby(['kategorie']) \
    .agg(sum_anzahl_pp=('anzahl_pp', 'sum')) \
    .sort_values('sum_anzahl_pp', ascending=False) 

myAgg.reset_index().head(3)

,kategorie,sum_anzahl_pp
0,Zweirad,44606.0


#### Step2: Datensätze exportieren


In [94]:
exp_file = imp_file.to_crs(epsg=2056)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\*JAHR*\quartierspiegel` als Shapefile


In [95]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/GdZ/export_zweiradabstellplaetze_p.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [96]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_zweiradabstellplaetze_p.gpkg


## ----------------------------------
## Spezialfall Mobility-Standorte. 

Früher hatten wir die Standorte auf dem Geoserver. Seit 2021 gibt es diese jedoch nicht mehr... Sie sind jedoch auf Opendata.swiss verfügbar.

Auf dem Geoserver gibt es sie unter https://geodaten.stzh.ch/geodatensatz/show?nbid=10763

ich habe die Daten manuell erstellt. Selektion von Provider Mobility und Region Zürich.

### Zweiradabstellplatz (12)

In [105]:
#Step1
url12 = ""


#Step2
typename=""

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

#Step 3
imp_file = gpd.read_file(url12)

imp_file.info()
#imp_file['bezeichnung'].unique()

DriverError: '' does not exist in the file system, and is not recognized as a supported dataset name.

In [93]:
#myAgg = data2betested.loc["2008-11-30":"2021-10-31"]
myAgg = imp_file\
    .groupby(['kategorie']) \
    .agg(sum_anzahl_pp=('anzahl_pp', 'sum')) \
    .sort_values('sum_anzahl_pp', ascending=False) 

myAgg.reset_index().head(3)

,kategorie,sum_anzahl_pp
0,Zweirad,44606.0


#### Step2: Datensätze exportieren


In [94]:
exp_file = imp_file.to_crs(epsg=2056)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\*JAHR*\quartierspiegel` als Shapefile


In [95]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/GdZ/export_zweiradabstellplaetze_p.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [96]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6556/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_zweiradabstellplaetze_p.gpkg



### ----------------------------------------
### Vorlage (X)

#### Step 1: Daten vom Geoportal beziehen und prüfen.

- url(X) --> Definiere die URL für den GEOJSON-Service
- typename --> Definiere den Layernamen. Kann an diversen Orten verwendet werden
- imp_name --> Leite den importierten Datensatznamen davon ab und ergänze wgs84
- exp_name --> ohne wgs84. soll heissen er ist per default lv95

In [139]:
urlX = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=X"

typename="X"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url4)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Bike-Park Allmend', 'Pumptrack Heuried'], dtype=object)

#### Step2: Datensätze exportieren


In [140]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [141]:
#exp_file.to_file(gis_ablagepfad+"export_"+typename+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_X.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [ ]:
exp_file.to_file("./output/shapefiles/"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackage/"+typename+date_today+".gpkg", driver='GPKG')
#gdf_poi_bikepark_view.plot()
print("GPKG exportiert nach: "+"./output/geopackage/"+typename+date_today+".gpkg")